In [11]:
# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# LangChain의 채팅용 언어 모델 초기화 함수입니다.
from langchain.chat_models import init_chat_model

# Google Cloud Vertex AI 의 채팅 모델(Gemini 등)을 사용하기 위한 클래스입니다.
from langchain_google_vertexai import ChatVertexAI

# 대화 메시지(history)를 상태로 다룰 수 있게 해주는 기본 상태 타입입니다.
from langgraph.graph.message import MessagesState

# ToolNode, tools_condition:
# - ToolNode: LLM 이 "도구를 써야 한다"고 판단했을 때 실제로 도구를 실행해 주는 노드
# - tools_condition: LLM 응답 안에 도구 호출이 있는지 없는지 보고,
#                    다음에 ToolNode 로 갈지/그냥 끝낼지 결정하는 조건 함수
from langgraph.prebuilt import ToolNode, tools_condition

# @tool 데코레이터로 파이썬 함수를 LangChain/LLM 이 쓸 수 있는 "도구"로 등록합니다.
from langchain_core.tools import tool

import warnings

# 구글 클라우드 aiplatform 패키지에서 발생하는 버전을 올리라는 경고(FutureWarning)를 무시합니다.
warnings.filterwarnings("ignore", category=FutureWarning, module="google.cloud.aiplatform")

# 아래는 사용할 LLM 설정 예시들입니다.
# (OpenAI, Gemnini 등 여러 방식 중 지금은 Vertex AI 기반 ChatVertexAI 를 사용)

# 실제로 사용할 채팅 모델(LLM)을 초기화합니다.
# - 여기서는 Google Cloud Vertex AI 의 Gemini 모델("gemini-2.5-flash")을 사용합니다.
# - project: GCP 프로젝트 ID
# - location: 리전(예: "asia-northeast3")
# - max_tokens: 한 번에 생성할 최대 토큰 수
llm = ChatVertexAI(
  model_name="gemini-2.5-flash-lite",  # 빠르고 저렴한 최신 모델
  project="ai-prompt-evaluator-489612", # 예: ai-prompt-evaluator-123
  location="us-central1",
  max_tokens=500
)


In [12]:
# MessagesState 를 상속해서, 대화형 상태를 위한 State 를 정의합니다.
# - MessagesState 안에는 보통 "messages" 라는 키에 대화 내역이 들어갑니다.
# - custom_stuff 같은 필드를 추가해서, 필요하면 더 많은 상태를 담을 수 있습니다.
class State(MessagesState):
  custom_stuff: str

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
graph_builder = StateGraph(State)


In [13]:
# @tool 데코레이터를 사용하면, 일반 파이썬 함수를
# LLM 이 호출할 수 있는 "도구(tool)" 로 등록할 수 있습니다.
# 이 함수는 "도시 이름을 받아, 그 도시의 날씨를 알려주는 도구" 역할을 합니다.
@tool
def get_weather(city: str):
  """Gets weather in city"""
  return f"The weather in {city} is sunny."

# llm.bind_tools(...): LLM 에게 "너는 이런 도구들을 쓸 수 있어" 라고 알려주는 단계입니다.
# - 이 예제에서는 get_weather 하나만 도구로 등록했습니다.
llm_with_tools = llm.bind_tools(tools=[get_weather])

# chatbot 노드는 현재까지의 대화 메시지(state["messages"])를
# 도구 사용이 가능한 LLM(llm_with_tools)에 보내고, 응답을 다시 messages 에 추가합니다.
# 만약 LLM 이 답변을 만들면서 "get_weather" 도구를 써야겠다고 판단하면,
# 응답 안에 도구 호출 정보가 포함되게 됩니다.
def chatbot(state: State):
  response = llm_with_tools.invoke(state["messages"])
  return {"messages": [response]}

  

In [14]:
# ToolNode 는 LLM 이 요청한 도구 호출을 실제로 실행해 주는 노드입니다.
# - tools 에 사용할 수 있는 도구 목록을 넘겨줍니다.
# - 예: LLM 이 응답에서 get_weather 를 호출하라고 지시하면,
#       ToolNode 가 get_weather 함수를 실제로 실행하고 결과를 state 에 반영합니다.
tool_node = ToolNode(
  tools=[get_weather],
)


In [ ]:
# 그래프에 노드들을 등록합니다.
# - "chatbot": LLM 에게 질문을 보내고 응답을 받는 노드
# - "tools": LLM 이 요청한 도구(get_weather)를 실제로 실행하는 ToolNode
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

# 실행 흐름을 정의합니다.
# 1) START -> chatbot: 먼저 챗봇 노드가 실행되어, LLM 응답을 만듭니다.
# 2) chatbot 이후에는 tools_condition 에 따라 분기합니다.
#    - LLM 응답 안에 "도구를 호출해야 한다"는 정보가 있으면 "tools" 노드로 이동
#    - 그렇지 않으면 바로 END 로 갈 수 있게 설계하는 패턴입니다.
# 3) tools 노드에서 실제로 도구를 실행한 뒤, 다시 chatbot 으로 돌아와
#    도구 결과를 바탕으로 후속 답변을 생성할 수 있습니다.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

# 지금까지 정의한 노드/간선들을 이용해서
# 실제로 실행 가능한 그래프 객체를 만듭니다.
graph = graph_builder.compile()

# 그래프를 한 번 실행해 봅니다.
# - user 가 "What is the weather in machupichu" 라고 물어보면,
# - chatbot → LLM 이 이 질문을 받고, get_weather 도구를 호출하고 싶어 할 수 있습니다.
# - 그 경우 tools_condition 이 이를 감지해서 tools 노드로 보내고,
#   ToolNode 가 get_weather("machupichu") 를 실행해 결과를 state 에 넣습니다.
# - 이후 다시 chatbot 으로 돌아와, 도구 결과를 활용한 최종 답변을 생성하게 됩니다.
graph.invoke(
  {
    "messages": [
      {
        "role": "user",
        "content": "What is the weather in machupichu",
      }
    ]
  }
)

# graph 객체 자체를 보면, 컴파일된 그래프 구조(CompiledStateGraph) 정보를 확인할 수 있습니다.
# graph

{'messages': [HumanMessage(content='대한민구 역대 최고의 왕은 누구인가요?', additional_kwargs={}, response_metadata={}, id='9ef31d1a-78d0-4de3-973e-fdbbbbc3c6a1'),
  AIMessage(content='죄송하지만, 저는 왕을 포함한 인물에 대한 정보를 가지고 있지 않습니다.', additional_kwargs={}, response_metadata={'is_blocked': False, 'safety_ratings': [], 'usage_metadata': {'prompt_token_count': 23, 'candidates_token_count': 18, 'total_token_count': 41, 'prompt_tokens_details': [{'modality': 1, 'token_count': 23}], 'candidates_tokens_details': [{'modality': 1, 'token_count': 18}], 'thoughts_token_count': 0, 'cached_content_token_count': 0, 'cache_tokens_details': []}, 'finish_reason': 'STOP', 'avg_logprobs': -0.4007400936550564, 'model_name': 'gemini-2.5-flash-lite'}, id='run--019cd801-49a1-7f83-996a-ab46677ff7be-0', usage_metadata={'input_tokens': 23, 'output_tokens': 18, 'total_tokens': 41, 'input_token_details': {'cache_read': 0}})]}